<a href="https://colab.research.google.com/github/B-Nanthana/HW04/blob/main/HW04_6810422016Nanthana.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# ==========================================
# ส่วนที่เพิ่มเข้ามา: ดึงข้อมูลจาก GitHub อัตโนมัติ
# ==========================================
# ระบุ URL ของไฟล์ Raw จาก GitHub ของคุณ
train_url = 'https://raw.githubusercontent.com/B-Nanthana/HW04/main/train.csv'
test_url = 'https://raw.githubusercontent.com/B-Nanthana/HW04/main/test.csv'

def clean_credit_data_complete(input_path, output_path, global_medians=None):
    # 1. โหลดข้อมูล (รองรับทั้ง Path ในเครื่องและ URL)
    df = pd.read_csv(input_path)

    # 2. จัดการค่า "ขยะ" เบื้องต้น
    df = df.replace(['_', '', ' ', '!@9#%8', '#F%$D@*&8'], np.nan)
    df = df.replace(r'^\s*$', np.nan, regex=True)
    df = df.replace(r'^_+$', np.nan, regex=True)

    # 3. ล้างคอลัมน์ตัวเลข
    numeric_cols = [
        'Age', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
        'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
        'Num_Credit_Inquiries', 'Annual_Income', 'Monthly_Inhand_Salary',
        'Changed_Credit_Limit', 'Outstanding_Debt', 'Total_EMI_per_month',
        'Amount_invested_monthly', 'Monthly_Balance'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('_', ''), errors='coerce')

    # 4. การ Match และเติมค่าว่างตามรายบุคคล (Group by Customer_ID)
    cat_to_fill = ['Name', 'SSN', 'Occupation', 'Type_of_Loan', 'Credit_Mix', 'Credit_History_Age', 'Payment_Behaviour']
    for col in cat_to_fill:
        if col in df.columns:
            df[col] = df.groupby('Customer_ID')[col].ffill().bfill()

    # 5. แก้ไขค่า Outlier
    def fix_with_median(df, column, condition, m_val):
        df.loc[condition, column] = m_val
        return df

    # คำนวณหรือใช้ค่า Median ที่ส่งมา
    if global_medians is None:
        global_medians = {col: df[col].median() for col in numeric_cols if col in df.columns}

    # จัดการ Outliers เบื้องต้น
    df = fix_with_median(df, 'Num_Bank_Accounts', (df['Num_Bank_Accounts'] < 0) | (df['Num_Bank_Accounts'] > 18), global_medians.get('Num_Bank_Accounts', 0))
    df = fix_with_median(df, 'Num_Credit_Card', (df['Num_Credit_Card'] > 10), global_medians.get('Num_Credit_Card', 0))
    df = fix_with_median(df, 'Interest_Rate', (df['Interest_Rate'] > 34), global_medians.get('Interest_Rate', 0))
    df = fix_with_median(df, 'Num_of_Loan', (df['Num_of_Loan'] < 0) | (df['Num_of_Loan'] > 9), global_medians.get('Num_of_Loan', 0))
    df = fix_with_median(df, 'Delay_from_due_date', (df['Delay_from_due_date'] < 1), global_medians.get('Delay_from_due_date', 0))
    df = fix_with_median(df, 'Num_of_Delayed_Payment', (df['Num_of_Delayed_Payment'] < 0) | (df['Num_of_Delayed_Payment'] > 28), global_medians.get('Num_of_Delayed_Payment', 0))
    df = fix_with_median(df, 'Num_Credit_Inquiries', (df['Num_Credit_Inquiries'] > 17), global_medians.get('Num_Credit_Inquiries', 0))
    df = fix_with_median(df, 'Age', (df['Age'] < 14), global_medians.get('Age', 0))

    # 6. จัดการค่าว่างหลังการ Match
    if 'Type_of_Loan' in df.columns:
        df['Type_of_Loan'] = df['Type_of_Loan'].fillna('Not Specified')

    for col in ['Name', 'SSN', 'Occupation', 'Credit_Mix', 'Payment_Behaviour']:
        if col in df.columns:
            df[col] = df[col].fillna('Unknown')

    for col in numeric_cols:
        if col in df.columns:
            df[col] = df[col].fillna(global_medians.get(col, 0))

    # 7. บันทึกผลลัพธ์ลงเครื่อง (ใน Colab)
    df.to_csv(output_path, index=False)
    print(f"เรียบร้อย! ทำความสะอาด {input_path} -> {output_path}")
    return global_medians

# --- ขั้นตอนการรัน ---

# 1. ล้างไฟล์ Train (ดึงจาก GitHub URL)
train_medians = clean_credit_data_complete(train_url, 'clean_train.csv')

# 2. ล้างไฟล์ Test (ดึงจาก GitHub URL) โดยใช้ค่า Median จาก Train
clean_credit_data_complete(test_url, 'clean_test.csv', global_medians=train_medians)

เรียบร้อย! ทำความสะอาด https://raw.githubusercontent.com/B-Nanthana/HW04/main/train.csv -> clean_train.csv
เรียบร้อย! ทำความสะอาด https://raw.githubusercontent.com/B-Nanthana/HW04/main/test.csv -> clean_test.csv


{'Age': 33.0,
 'Num_Bank_Accounts': 6.0,
 'Num_Credit_Card': 5.0,
 'Interest_Rate': 13.0,
 'Num_of_Loan': 3.0,
 'Delay_from_due_date': 18.0,
 'Num_of_Delayed_Payment': 14.0,
 'Num_Credit_Inquiries': 6.0,
 'Annual_Income': 37550.74,
 'Monthly_Inhand_Salary': 3086.7566666666667,
 'Changed_Credit_Limit': 9.4,
 'Outstanding_Debt': 1166.155,
 'Total_EMI_per_month': 68.85671805539175,
 'Amount_invested_monthly': 136.04383222758537,
 'Monthly_Balance': 336.5887577751173}

In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import f1_score, classification_report

# ==========================================
# 1. โหลดข้อมูล
# ==========================================
df = pd.read_csv('clean_train.csv')

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================

# ก. แปลง Credit_History_Age เป็นจำนวนเดือน
def extract_months(text):
    if pd.isna(text) or text == 'Unknown': return 0
    try:
        parts = text.split()
        return (int(parts[0]) * 12) + int(parts[3])
    except: return 0

df['Credit_History_Age_Months'] = df['Credit_History_Age'].apply(extract_months)

# ข. จัดการ Outliers (Capping)
df['Num_Bank_Accounts'] = df['Num_Bank_Accounts'].clip(0, 15)
df['Num_Credit_Card'] = df['Num_Credit_Card'].clip(0, 12)
df['Num_of_Loan'] = df['Num_of_Loan'].clip(0, 10)
df['Interest_Rate'] = df['Interest_Rate'].clip(0, 40)

# ค. Log Transformation
for col in ['Annual_Income', 'Outstanding_Debt', 'Monthly_Balance']:
    df[col] = np.log1p(df[col].clip(lower=0))

# ==========================================
# 3. การเลือก Feature และเตรียมข้อมูล
# ==========================================
features = [
    'Age', 'Annual_Income', 'Monthly_Inhand_Salary',
    'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate',
    'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment',
    'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Credit_Mix',
    'Outstanding_Debt', 'Total_EMI_per_month', 'Amount_invested_monthly',
    'Monthly_Balance', 'Payment_Behaviour',
    'Credit_History_Age_Months'
]

X = df[features].copy()
y = df['Credit_Score']

# แปลง Categorical เป็น One-Hot
X = pd.get_dummies(X, columns=['Credit_Mix', 'Payment_Behaviour'])

# แปลง Target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# แบ่งข้อมูล 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_full_scaled = scaler.transform(X) # สำหรับคำนวณ Full Set

# ==========================================
# 4. สร้างและเทรน Model
# ==========================================

# Logistic Regression
lr = LogisticRegression(C=10.0, max_iter=10000, class_weight='balanced', solver='saga', random_state=42)
lr.fit(X_train_scaled, y_train)

# Naive Bayes
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)

# ==========================================
# 5. แสดงผลลัพธ์ (Train, Test, Full)
# ==========================================

def calculate_f1(model, X_scaled, y_true):
    y_pred = model.predict(X_scaled)
    return f1_score(y_true, y_pred, average='weighted')

# คำนวณค่า F1 ของ Logistic Regression
f1_lr_train = calculate_f1(lr, X_train_scaled, y_train)
f1_lr_test = calculate_f1(lr, X_test_scaled, y_test)
f1_lr_full = calculate_f1(lr, X_full_scaled, y_encoded)

# คำนวณค่า F1 ของ Naive Bayes
f1_nb_train = calculate_f1(nb, X_train_scaled, y_train)
f1_nb_test = calculate_f1(nb, X_test_scaled, y_test)
f1_nb_full = calculate_f1(nb, X_full_scaled, y_encoded)

print("="*85)
print(f"{'Metric (Weighted F1-Score)':<35} | {'Logistic Regression':<20} | {'Naive Bayes':<20}")
print("-" * 85)
print(f"{'F1 Score: Train Set (80%)':<35} | {f1_lr_train:.4f}             | {f1_nb_train:.4f}")
print(f"{'F1 Score: Test Set (20%)':<35} | {f1_lr_test:.4f}             | {f1_nb_test:.4f}")
print(f"{'F1 Score: Full Dataset (100%)':<35} | {f1_lr_full:.4f}             | {f1_nb_full:.4f}")
print("="*85)

# ==========================================
# 6. พยากรณ์ข้อมูลลงไฟล์ results.csv
# ==========================================
df_final = pd.read_csv('clean_test.csv')
X_test_temp = df_final.copy()

# Feature Engineering ชุดเดิม
X_test_temp['Credit_History_Age_Months'] = X_test_temp['Credit_History_Age'].apply(extract_months)
X_test_temp['Num_Bank_Accounts'] = X_test_temp['Num_Bank_Accounts'].clip(0, 15)
X_test_temp['Num_Credit_Card'] = X_test_temp['Num_Credit_Card'].clip(0, 12)
X_test_temp['Num_of_Loan'] = X_test_temp['Num_of_Loan'].clip(0, 10)
X_test_temp['Interest_Rate'] = X_test_temp['Interest_Rate'].clip(0, 40)
for col in ['Annual_Income', 'Outstanding_Debt', 'Monthly_Balance']:
    X_test_temp[col] = np.log1p(X_test_temp[col].clip(lower=0))

X_for_pred = X_test_temp[features].copy()
X_for_pred = pd.get_dummies(X_for_pred, columns=['Credit_Mix', 'Payment_Behaviour'])
X_for_pred = X_for_pred.reindex(columns=X.columns, fill_value=0)

X_for_pred_scaled = scaler.transform(X_for_pred)
df_final['Credit_Score'] = le.inverse_transform(lr.predict(X_for_pred_scaled))

df_final.to_csv('results.csv', index=False)
print("\n[สำเร็จ] คำนวณ F1 ครบทุกส่วนและสร้างไฟล์ 'results.csv' แล้ว")

Metric (Weighted F1-Score)          | Logistic Regression  | Naive Bayes         
-------------------------------------------------------------------------------------
F1 Score: Train Set (80%)           | 0.6657             | 0.6566
F1 Score: Test Set (20%)            | 0.6628             | 0.6578
F1 Score: Full Dataset (100%)       | 0.6651             | 0.6568

[สำเร็จ] คำนวณ F1 ครบทุกส่วนและสร้างไฟล์ 'results.csv' แล้ว
